# 🤖 Notebook 09 - Model Serialization
**Project:** AI-Driven Citizen Grievance Analysis

**Purpose:** Serialize trained models and tokenizers for production deployment with versioning metadata.

---
**Inputs:** Trained models + evaluation metrics from Notebook 08

**Outputs:**
- `../models/final_models/sentiment_model/`
- `../models/final_models/sentiment_metadata.json`
- `../models/final_models/department_model/`
- `../models/final_models/department_metadata.json`


##  Imports & Config

In [1]:
import json
import torch
import warnings
from pathlib import Path
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')

class Config:
    BASE_DIR          = Path('../')
    MODELS_DIR        = BASE_DIR / 'models'
    EVAL_DIR          = BASE_DIR / 'evaluation'
    FINAL_MODELS_DIR  = MODELS_DIR / 'final_models'
    SENTIMENT_MODEL   = MODELS_DIR / 'sentiment_model'
    DEPARTMENT_MODEL  = MODELS_DIR / 'department_classifier'
    MODEL_NAME        = 'distilroberta-base'
    MAX_LENGTH        = 128
    DEVICE            = 'cuda' if torch.cuda.is_available() else 'cpu'
    SENTIMENT_LABELS  = {0: 'positive', 1: 'neutral', 2: 'negative', 3: 'critical'}
    DEPARTMENT_LABELS = {
        0: 'environment', 1: 'non_compliant', 2: 'social_health_services', 3: 'transport'
    }

Config.FINAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)
print('Configuration ready')
print(f'  Device           : {Config.DEVICE}')
print(f'  Final Models Dir : {Config.FINAL_MODELS_DIR}')


Configuration ready
  Device           : cuda
  Final Models Dir : ../models/final_models


##  Load Evaluation Metrics

In [2]:
def load_metrics(filepath):
    try:
        with open(filepath) as f:
            m = json.load(f)
        print(f'Loaded metrics: {filepath}')
        return m
    except FileNotFoundError:
        print(f'[WARN] Metrics file not found: {filepath}')
        print('  Using placeholder. Run Notebook 08 first.')
        return {'accuracy': 0.0, 'f1_score': 0.0, 'precision': 0.0, 'recall': 0.0}


sentiment_metrics  = load_metrics(Config.EVAL_DIR / 'sentiment_metrics.json')
department_metrics = load_metrics(Config.EVAL_DIR / 'department_metrics.json')

print(f'\nSentiment  -> Accuracy: {sentiment_metrics["accuracy"]:.4f} | F1: {sentiment_metrics["f1_score"]:.4f}')
print(f'Department -> Accuracy: {department_metrics["accuracy"]:.4f} | F1: {department_metrics["f1_score"]:.4f}')


Loaded metrics: ../evaluation/sentiment_metrics.json
Loaded metrics: ../evaluation/department_metrics.json

Sentiment  -> Accuracy: 1.0000 | F1: 1.0000
Department -> Accuracy: 0.0904 | F1: 0.0150


##  Load Models

In [3]:
def load_model(model_path, model_name, num_labels, label):
    print(f'Loading {label} model...')
    try:
        model     = AutoModelForSequenceClassification.from_pretrained(
            str(model_path), num_labels=num_labels)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        print(f'  Loaded from: {model_path}')
    except Exception as e:
        print(f'  Saved model unavailable ({e})')
        print(f'  Using base model: {model_name}')
        model     = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    return model, tokenizer


sentiment_model,  sentiment_tokenizer  = load_model(
    Config.SENTIMENT_MODEL,  Config.MODEL_NAME, 4, 'Sentiment')
department_model, department_tokenizer = load_model(
    Config.DEPARTMENT_MODEL, Config.MODEL_NAME, 4, 'Department')


Loading Sentiment model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  Loaded from: ../models/sentiment_model
Loading Department model...
  Saved model unavailable (Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../models/department_classifier'. Use `repo_type` argument if needed.)
  Using base model: distilroberta-base


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Serialize Models

In [4]:
class ModelSerializer:
    """Save models and tokenizers for production"""

    @staticmethod
    def serialize(model, tokenizer, dest_dir, task, num_labels, labels_map, metrics):
        dest = Config.FINAL_MODELS_DIR / dest_dir
        model.save_pretrained(str(dest))
        tokenizer.save_pretrained(str(dest))

        metadata = {
            'model_name'  : Config.MODEL_NAME,
            'task'        : task,
            'num_labels'  : num_labels,
            'max_length'  : Config.MAX_LENGTH,
            'labels'      : labels_map,
            'created_at'  : datetime.now().isoformat(),
            'version'     : '1.0.0',
            'accuracy'    : metrics['accuracy'],
            'f1_score'    : metrics['f1_score'],
            'precision'   : metrics['precision'],
            'recall'      : metrics['recall'],
        }
        meta_path = Config.FINAL_MODELS_DIR / f'{dest_dir.replace("_model","")}_metadata.json'
        with open(meta_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        print(f'Saved model    -> {dest}')
        print(f'Saved metadata -> {meta_path}')
        return dest, metadata


print('Serializing sentiment model...')
s_path, s_meta = ModelSerializer.serialize(
    sentiment_model, sentiment_tokenizer,
    'sentiment_model', 'sentiment_classification',
    4, Config.SENTIMENT_LABELS, sentiment_metrics
)

print('\nSerializing department model...')
d_path, d_meta = ModelSerializer.serialize(
    department_model, department_tokenizer,
    'department_model', 'department_classification',
    4, Config.DEPARTMENT_LABELS, department_metrics
)


Serializing sentiment model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model    -> ../models/final_models/sentiment_model
Saved metadata -> ../models/final_models/sentiment_metadata.json

Serializing department model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model    -> ../models/final_models/department_model
Saved metadata -> ../models/final_models/department_metadata.json


## Verify Saved Models

In [5]:
def verify_model(model_dir, num_labels, label):
    print(f'Verifying {label}...')
    try:
        m = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
        t = AutoTokenizer.from_pretrained(str(model_dir))
        m.eval()
        inputs = t('Test complaint text', return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            pred = torch.argmax(m(**inputs).logits, dim=1).item()
        print(f'  PASSED - test prediction index: {pred}')
        return True
    except Exception as e:
        print(f'  FAILED: {e}')
        return False


ok1 = verify_model(s_path, 4, 'Sentiment Model')
ok2 = verify_model(d_path, 4, 'Department Model')

print('\n' + '='*50)
if ok1 and ok2:
    print('ALL MODELS VERIFIED SUCCESSFULLY')
else:
    print('WARNING: Some models failed verification')
print('='*50)


Verifying Sentiment Model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  PASSED - test prediction index: 2
Verifying Department Model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  PASSED - test prediction index: 1

ALL MODELS VERIFIED SUCCESSFULLY


## Summary

Both models serialized using HuggingFace `save_pretrained()`.

| Model | Path | Labels | Version |
|-------|------|--------|---------|
| Sentiment | `models/final_models/sentiment_model/` | 4 | 1.0.0 |
| Department | `models/final_models/department_model/` | 4 | 1.0.0 |

**Department Labels:** `environment`, `non_compliant`, `social_health_services`, `transport`

**Next:** Run `10_fastapi_development.ipynb`
